In [1]:
!pip install requests transformers torch -q
print("Libraries installed!")

Libraries installed!


In [2]:
# Cell 2: Setup API and define ESG messages to triage

HF_TOKEN = "hf_..."  # paste your token here

# 5 sample ESG messages for triage
esg_messages = [
    {
        "id": "MSG001",
        "text": "Our factory in Vietnam has been discharging untreated wastewater into the local river for the past 6 months. Workers have complained but management ignored them.",
        "expected_category": "Environmental"
    },
    {
        "id": "MSG002",
        "text": "The board approved a $50M share buyback while cutting employee health benefits by 30%. Several executives received bonuses exceeding $2M despite missing profit targets.",
        "expected_category": "Governance"
    },
    {
        "id": "MSG003",
        "text": "Child labor has been reported at two of our Tier-2 suppliers in Bangladesh. Local NGOs have documented children aged 12-14 working 10-hour shifts.",
        "expected_category": "Social"
    },
    {
        "id": "MSG004",
        "text": "Carbon emissions from our logistics operations increased by 18% last year, exceeding our stated net-zero commitment targets by a significant margin.",
        "expected_category": "Environmental"
    },
    {
        "id": "MSG005",
        "text": "Three senior managers have resigned citing a toxic workplace culture. Anonymous surveys reveal 67% of employees feel psychologically unsafe raising concerns.",
        "expected_category": "Social"
    }
]

print(f"Loaded {len(esg_messages)} ESG messages for triage")
for msg in esg_messages:
    print(f"  {msg['id']}: {msg['text'][:60]}...")

Loaded 5 ESG messages for triage
  MSG001: Our factory in Vietnam has been discharging untreated wastew...
  MSG002: The board approved a $50M share buyback while cutting employ...
  MSG003: Child labor has been reported at two of our Tier-2 suppliers...
  MSG004: Carbon emissions from our logistics operations increased by ...
  MSG005: Three senior managers have resigned citing a toxic workplace...


In [3]:
# Cell 3: Part A - LLM-based ESG triage using HuggingFace Inference API

import requests
import json
import time

# Using Mistral-7B via HuggingFace Inference API (free tier)
API_URL = "https://api-inference.huggingface.co/models/mistralai/Mistral-7B-Instruct-v0.3"
headers = {"Authorization": f"Bearer {HF_TOKEN}"}

# Improved prompt template
PROMPT_TEMPLATE = """You are an ESG (Environmental, Social, Governance) compliance analyst.
Analyse the following message and return a JSON response with exactly these fields:
- category: one of "Environmental", "Social", or "Governance"
- severity: one of "Low", "Medium", "High", or "Critical"
- urgency: one of "Routine", "Urgent", or "Immediate"
- summary: a one-sentence summary of the issue (max 20 words)
- recommended_action: a specific next step for the compliance team

Message: {message}

Respond with valid JSON only. No explanation, no markdown, just the JSON object."""

def triage_message_llm(message_text):
    prompt = PROMPT_TEMPLATE.format(message=message_text)

    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": 250,
            "temperature": 0.1,
            "return_full_text": False
        }
    }

    try:
        response = requests.post(API_URL, headers=headers, json=payload, timeout=60)
        result = response.json()

        if isinstance(result, list) and len(result) > 0:
            raw_text = result[0].get("generated_text", "")
            # Extract JSON from response
            start = raw_text.find("{")
            end = raw_text.rfind("}") + 1
            if start != -1 and end > start:
                json_str = raw_text[start:end]
                return json.loads(json_str)
        return {"error": "Could not parse response", "raw": str(result)[:200]}

    except Exception as e:
        return {"error": str(e)}

# Run triage on all messages
print("=" * 60)
print("LLM-BASED ESG TRIAGE RESULTS")
print("=" * 60)

llm_results = []
for msg in esg_messages:
    print(f"\nProcessing {msg['id']}...")
    result = triage_message_llm(msg["text"])
    result["message_id"] = msg["id"]
    result["expected_category"] = msg["expected_category"]
    llm_results.append(result)
    print(json.dumps(result, indent=2))
    time.sleep(2)  # Rate limit pause

print("\n✅ LLM triage complete!")

LLM-BASED ESG TRIAGE RESULTS

Processing MSG001...
{
  "error": "HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/mistralai/Mistral-7B-Instruct-v0.3 (Caused by NameResolutionError(\"<urllib3.connection.HTTPSConnection object at 0x7983141ee5d0>: Failed to resolve 'api-inference.huggingface.co' ([Errno -2] Name or service not known)\"))",
  "message_id": "MSG001",
  "expected_category": "Environmental"
}

Processing MSG002...
{
  "error": "HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/mistralai/Mistral-7B-Instruct-v0.3 (Caused by NameResolutionError(\"<urllib3.connection.HTTPSConnection object at 0x798315e01580>: Failed to resolve 'api-inference.huggingface.co' ([Errno -2] Name or service not known)\"))",
  "message_id": "MSG002",
  "expected_category": "Governance"
}

Processing MSG003...
{
  "error": "HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): 

In [4]:
# Cell 4: Part B - Zero-shot classification baseline for comparison

from transformers import pipeline

print("Loading zero-shot classifier...")
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# ESG categories for classification
candidate_labels = ["Environmental", "Social", "Governance"]

# Severity keywords for rule-based scoring
severity_keywords = {
    "Critical": ["child labor", "illegal", "criminal", "death", "toxic", "untreated wastewater"],
    "High": ["resignation", "harassment", "discrimination", "violation", "unsafe"],
    "Medium": ["complaint", "concern", "exceeded", "missed target", "reduced"],
    "Low": ["minor", "routine", "small", "marginal"]
}

def get_rule_based_severity(text):
    text_lower = text.lower()
    for level, keywords in severity_keywords.items():
        if any(kw in text_lower for kw in keywords):
            return level
    return "Medium"

print("\n" + "=" * 60)
print("ZERO-SHOT BASELINE RESULTS")
print("=" * 60)

baseline_results = []
for msg in esg_messages:
    # Zero-shot category classification
    zs_result = classifier(msg["text"], candidate_labels)
    predicted_category = zs_result["labels"][0]
    confidence = round(zs_result["scores"][0], 3)

    # Rule-based severity
    severity = get_rule_based_severity(msg["text"])

    result = {
        "message_id": msg["id"],
        "predicted_category": predicted_category,
        "confidence": confidence,
        "all_scores": {
            label: round(score, 3)
            for label, score in zip(zs_result["labels"], zs_result["scores"])
        },
        "severity_rule_based": severity,
        "expected_category": msg["expected_category"]
    }
    baseline_results.append(result)
    print(f"\n{msg['id']}: {predicted_category} (confidence: {confidence})")
    print(f"  Severity (rule-based): {severity}")
    print(f"  All scores: {result['all_scores']}")

print("\n✅ Baseline classification complete!")

Loading zero-shot classifier...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


ZERO-SHOT BASELINE RESULTS

MSG001: Environmental (confidence: 0.421)
  Severity (rule-based): Critical
  All scores: {'Environmental': 0.421, 'Social': 0.391, 'Governance': 0.188}

MSG002: Governance (confidence: 0.586)
  Severity (rule-based): Medium
  All scores: {'Governance': 0.586, 'Social': 0.34, 'Environmental': 0.073}

MSG003: Social (confidence: 0.589)
  Severity (rule-based): Critical
  All scores: {'Social': 0.589, 'Governance': 0.304, 'Environmental': 0.107}

MSG004: Environmental (confidence: 0.452)
  Severity (rule-based): Medium
  All scores: {'Environmental': 0.452, 'Social': 0.424, 'Governance': 0.124}

MSG005: Social (confidence: 0.681)
  Severity (rule-based): Critical
  All scores: {'Social': 0.681, 'Governance': 0.231, 'Environmental': 0.088}

✅ Baseline classification complete!


In [5]:
# Cell 5: Compare LLM vs Baseline and save results

print("=" * 60)
print("COMPARISON: LLM vs ZERO-SHOT BASELINE")
print("=" * 60)

correct_llm = 0
correct_baseline = 0

print(f"\n{'ID':<8} {'Expected':<15} {'LLM':<15} {'Baseline':<15} {'LLM✓':<8} {'Base✓'}")
print("-" * 70)

for llm, base in zip(llm_results, baseline_results):
    expected = llm["expected_category"]
    llm_cat = llm.get("category", "N/A")
    base_cat = base["predicted_category"]

    llm_correct = "✅" if llm_cat == expected else "❌"
    base_correct = "✅" if base_cat == expected else "❌"

    if llm_cat == expected:
        correct_llm += 1
    if base_cat == expected:
        correct_baseline += 1

    print(f"{llm['message_id']:<8} {expected:<15} {llm_cat:<15} {base_cat:<15} {llm_correct:<8} {base_correct}")

total = len(esg_messages)
print("-" * 70)
print(f"\nAccuracy — LLM: {correct_llm}/{total} ({correct_llm/total*100:.0f}%)")
print(f"Accuracy — Baseline: {correct_baseline}/{total} ({correct_baseline/total*100:.0f}%)")

# Save experiment log to JSON
experiment_log = {
    "experiment": "ESG Message Triage",
    "date": "2026",
    "model_llm": "mistralai/Mistral-7B-Instruct-v0.3",
    "model_baseline": "facebook/bart-large-mnli (zero-shot)",
    "messages_tested": total,
    "llm_accuracy": f"{correct_llm/total*100:.0f}%",
    "baseline_accuracy": f"{correct_baseline/total*100:.0f}%",
    "llm_results": llm_results,
    "baseline_results": baseline_results
}

with open("experiment_log.json", "w") as f:
    json.dump(experiment_log, f, indent=2)

print("\n✅ Experiment log saved to experiment_log.json")
print("📁 Download this file and upload to your GitHub repo!")

COMPARISON: LLM vs ZERO-SHOT BASELINE

ID       Expected        LLM             Baseline        LLM✓     Base✓
----------------------------------------------------------------------
MSG001   Environmental   N/A             Environmental   ❌        ✅
MSG002   Governance      N/A             Governance      ❌        ✅
MSG003   Social          N/A             Social          ❌        ✅
MSG004   Environmental   N/A             Environmental   ❌        ✅
MSG005   Social          N/A             Social          ❌        ✅
----------------------------------------------------------------------

Accuracy — LLM: 0/5 (0%)
Accuracy — Baseline: 5/5 (100%)

✅ Experiment log saved to experiment_log.json
📁 Download this file and upload to your GitHub repo!
